Do some leave one out test of limits to what template should be used.
Something like:
- pick one class.
- Go through each SN.
- Loop through all WarpSources of this class except the SN.
- Fit and store fit quality results.
- Possibly make a plot of the fits, possibly also of the unwarped templates.
- Make aggregate plots of fit quality vs cuts (chi2, bands, dof, fit good ...)
- Use these to decide on limits for which WarpSources to use for simulations.
- (Then rerun and create figures)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker 
import pickle
#import json
import re
import pymongo
import sncosmo
from warptemplate import get_ztftable_from_ampel, get_warpedTimeSeriesModel
#from warptemplate import get_ztftable_from_ampel, get_template_correction
from typing import Literal
from collections import Counter

In [ ]:
# References warpclasses 
classnbr = 8

In [ ]:
warpclasses = [
    'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T',
    'SN Ib/c', 'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic',
    'SLSN-II']
parentclasses = {
    'SN Ic':['SN Ic', 'SN Ic-BL', 'SN Ic-pec', 'SN Icn'], 
    'SN Ib/c':['SN Ibc', 'SN Ib', 'SN Ic', 'SN Ibn', 'SN Ic-BL', 'SN Ic-pec', 'SN Icn'], 
    'SN II': ['SN II','SN IIP', 'SN II-pec', 'SN IIL']
}

In [ ]:
bts_tomatch = [warpclasses[classnbr]]
if warpclasses[classnbr] in parentclasses:
    bts_tomatch = parentclasses[warpclasses[classnbr]]

In [ ]:
storefile = '/Users/jnordin/tmp/warps_'+re.sub(r'/', '', warpclasses[classnbr])+'.pkl'

In [ ]:
with open(storefile, 'rb') as file:
    infocol = pickle.load(file)

In [ ]:
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase    # Only final lc. try bts_ipacfp_strictbase_full for all alerts

In [ ]:
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')

### Find subset of BTS SNe matching target class

In [ ]:
df_bts = df_bts[( df_bts['type'].isin(bts_tomatch) )]

In [ ]:
len(df_bts)

In [ ]:
df_bts = df_bts.iloc[0:20]

In [ ]:
# datacollect = {}

In [ ]:
count=0
for k, sninfo in df_bts.iterrows():
    print(count)
    if sninfo['ZTFID'] in datacollect:
        print('... done')
        continue
    count += 1
    if count>10:
        break
    
    tab = get_ztftable_from_ampel( sninfo['ZTFID'], db, 
                              redshift=float(sninfo['redshift']), 
                              type = sninfo['type'] )
    datacollect[sninfo['ZTFID']] = {
        'sninfo': dict(sninfo),
        'tab': tab
    }
    if len(tab)==0:
        print('{} ... no baseline corr info.'.format(sninfo['ZTFID']))
        continue
    if len(set(tab['band']))<2:
        print('{} ... less than two bands.'.format(sninfo['ZTFID']))
        continue
    print('START RUNNING', sninfo['ZTFID'])
    # Start with all the warp fits
    warped_fits = {}
    for basesn, fitinfo in infocol.items():
#        print('still at', sninfo['ZTFID'])
        if basesn==sninfo['ZTFID']:
            print('... do not fit to the same sn warp models.')
            continue
    
        # Start looping through all models
        for modelname, fitdata in fitinfo.items():
            if not fitdata['success']:
#                print('... did not work, skipping', modelname)
                continue
#            print('Running fit wit corr to', modelname)
    
            wm = get_warpedTimeSeriesModel(
                name=basesn+'_'+modelname, 
                original_template_name=modelname, 
                warpdata=fitdata,
                z=float(sninfo['redshift']),
                original_template_version=None
            )
            fitprop = ['t0', 'amplitude', 'hostebv']
    
            try:
                wresult, wfitted_model = sncosmo.fit_lc(
                    tab, wm,
                    fitprop,  # parameters of model to vary
                )
            except (ValueError, RuntimeError):
#                print('... fit error')
                continue
            if not wresult['success']:
#                print('... fit fail')
                continue
            warped_fits[basesn+'_'+modelname] = {
                'sncosmo_result': wresult,
                'sncosmo_model': wfitted_model,
                'base_sn':basesn,
                    'base_template': modelname,
            }
            # Retain warp info for result comparisons
            warped_fits[basesn+'_'+modelname].update(
                {k:fitdata[k] for k in ['dps_init', 'hostebv', 'success', 'ndof', 'absmag', 'chidof', 'lceval']}
            )
    datacollect[sninfo['ZTFID']]['warped_fits']=warped_fits
    if count>(runcount*(run+1)):
        print('run done')
        break


In [ ]:
len(datacollect)

### Evaluate fits
We have now fit each BTS SN with each warp source.
We now wish to parse the success of these. The goal is to define
a subset of warp models, combine with mininmmal requirements on data quality, 
such that an as large as possible fraction of the BTS SN (of this class)
can be accurately modeled. 
Note: we probably need to create some sort of intrinsic dispersion for each class. Maybe fit in a second step? 

Steps:
- Find the best chi2dof for each BTS SN, look for cases which cannot be explained at all. Study these separately? Less data, distance, bands? Weird? Bad calibration?
- Find whether there are warp sources that are never really any of the top fits. These can then be removed for subsequent runs to speed things up and especially fits.
- Estimate an intrinsic dispersion to be used with this class?
- Rerun and check frequence of good fits, to be used as weights in simulations.
- Make some nice plots? 

In [ ]:
dfsort = df.sort_values('chidof',ascending=True).groupby('warp_sn').head(1)

In [ ]:
# First question - find out which BTS SNe cannot be fit at all
# by any template. These will only distort things.
# So look at the ones with min chi2dof above some threshold 
# then we will remove these and do the same for warp sn and template.
df.groupby('bts_sn').describe()

In [ ]:
#sns.set_theme(style="ticks")
f, ax = plt.subplots(figsize=(7, 5))

sns.histplot(
    dfsort,
    x='chidof', hue="warp_template",
    multiple="stack",
#    palette="light:m_r",
    edgecolor=".3",
    linewidth=.5,
    legend=False,
)
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())


### Evaluate fits

In [ ]:
# Make a hist of chi and/or chidof for all fits
# Possibly distinguish fits by different base models
# Similarly explore fit parameter values? Can they give something?
# To be used to decide on some cuts? 
# fig = sncosmo.plot_lc(tab, model=wfitted_model, errors=wresult.errors)

In [ ]:
warped_fits['ZTF18abwlupf_v19-1999dn-corr']

In [ ]:
fitsummary = [
    {
        'fit_chisq': val['sncosmo_result']['chisq'], 
        'fit_ndof': val['sncosmo_result']['ndof'], 
        'fit_ebv': val['sncosmo_result']['parameters'][3],
        'base_sn': val['base_sn'],
        'base_template': val['base_template'],
        'warp_dps': val['dps_init'],
        'warp_ebv': val['hostebv'],
        'warp_success': val['success'],
        'warp_peakndof': val['ndof'],
        'warp_peakchidof': val['chidof'],
    } 
    for key, val in warped_fits.items()
]

In [ ]:
df = pd.DataFrame.from_dict(fitsummary)
df['fit_chidof'] = df['fit_chisq'] / df['fit_ndof'] 

In [ ]:
sns.set_theme(style="ticks")
f, ax = plt.subplots(figsize=(7, 5))
#sns.despine(f)

sns.histplot(
    df,
    x='fit_chidof', hue="base_template",
    multiple="stack",
#    palette="light:m_r",
    edgecolor=".3",
    linewidth=.5,
#    log_scale=True,
    legend=False,
)
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
#ax.set_xticks([500, 1000, 2000, 5000, 10000])

# Does not look like any particular template has low/high chisq

In [ ]:
sns.set_theme(style="ticks")
f, ax = plt.subplots(figsize=(7, 5))
#sns.despine(f)

sns.histplot(
    df,
    x='fit_chidof', hue="base_sn",
    multiple="stack",
#    palette="light:m_r",
    edgecolor=".3",
    linewidth=.5,
#    log_scale=True,
    legend=False,
)
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
#ax.set_xticks([500, 1000, 2000, 5000, 10000])

# Similarly for SN - a little weird?

In [ ]:
# Explore whether some relation between warp input data and chi
sns.relplot(x="fit_chidof", y="fit_ebv", hue="base_template", size="base_sn",
        #    sizes=(40, 400), 
            alpha=.5, palette="muted",
            height=6, data=df, legend=False)

In [ ]:
# Explore whether some relation between warp input data and chi
# not fit_ebv, warp_dps, warp_ebv, warp_peakchidof,warp_peakndof - not really 
sns.relplot(x="fit_chidof", y="warp_peakndof", hue="base_template", size="base_sn",
        #    sizes=(40, 400), 
            alpha=.5, palette="muted",
            height=6, data=df, legend=False)

### Reduce fits
Too many things to study at once.
Only retain the lowst chisq for either each base_sn or each base_template.
Which is better?
Also needs to keep track on whether some are consistently being removed? 

In [ ]:
# A1: Only keep min chisq for each base sn
#dfsort = df.sort_values('fit_chidof',ascending=True).groupby('base_sn').head(1)
# A1 at one instance did not retain good fits? 
# A2: Only keep min chisq for each base template
dfsort = df.sort_values('fit_chidof',ascending=True).groupby('base_template').head(1)

In [ ]:
dfsort

In [ ]:
keepcombos = [
    sn+'_'+t for sn, t in zip (dfsort['base_sn'], dfsort['base_template'])
]

In [ ]:
warpfits_cut = {
    keepcomb: warped_fits[keepcomb]
    for keepcomb in keepcombos
}

In [ ]:
len(warpfits_cut)

## Plot the fits, in one band, to one SN

In [ ]:
plotsn = 'ZTF19aaeopgw'
# We clearly need to think about which sne to include both for template
# as well as for verification. 
# Bad lc: 'ZTF18abwlupf'
# No good fits: ZTF18acnnevs ZTF19aaadwfi
# Good: ZTF18acsxwdi 

In [ ]:
datacollect.keys()

In [ ]:
plotband = 'ztfg'

In [ ]:
tab = get_ztftable_from_ampel( plotsn, db, 
                              redshift=float(df_bts[df_bts['ZTFID']=='ZTF18abwlupf']['redshift'].iloc[0]),
                            type = df_bts[df_bts['ZTFID']=='ZTF18abwlupf']['type'].iloc[0] 
)

In [ ]:
ptab = tab[tab['band']==plotband]

In [ ]:
warped_fits = datacollect[plotsn]['warped_fits']

In [ ]:
fitsummary = [
    {
        'fit_chisq': val['sncosmo_result']['chisq'], 
        'fit_ndof': val['sncosmo_result']['ndof'], 
        'fit_ebv': val['sncosmo_result']['parameters'][3],
        'base_sn': val['base_sn'],
        'base_template': val['base_template'],
        'warp_dps': val['dps_init'],
        'warp_ebv': val['hostebv'],
        'warp_success': val['success'],
        'warp_peakndof': val['ndof'],
        'warp_peakchidof': val['chidof'],
    } 
    for key, val in warped_fits.items()
]

In [ ]:
df = pd.DataFrame.from_dict(fitsummary)
df['fit_chidof'] = df['fit_chisq'] / df['fit_ndof'] 

In [ ]:
# A1: Only keep min chisq for each base sn
#dfsort = df.sort_values('fit_chidof',ascending=True).groupby('base_sn').head(1)
# A1 at one instance did not retain good fits? 
# A2: Only keep min chisq for each base template
dfsort = df.sort_values('fit_chidof',ascending=True).groupby('base_template').head(1)

In [ ]:
keepcombos = [
    sn+'_'+t for sn, t in zip (dfsort['base_sn'], dfsort['base_template'])
]

In [ ]:
warpfits_cut = {
    keepcomb: warped_fits[keepcomb]
    for keepcomb in keepcombos
}

In [ ]:
len(warpfits_cut)

In [ ]:
# Key parameters?
# A limit of 10 seems a region where most fits are reasonable
max_chidof = 30

In [ ]:
plt.plot(ptab['time'], ptab['flux'], 'o')
timeplot = np.arange( min(ptab['time']), max(ptab['time']), 1)
#for key, fitres in warped_fits.items():
#    break
for key, fitres in warpfits_cut.items():    
#    print(key)
    #print(fitres)
    #break
    if fitres['sncosmo_result']['chisq'] / fitres['sncosmo_result']['ndof'] > max_chidof:
        continue
    
    m = fitres['sncosmo_model']

    # Plots, assuming zp constant
    plt.plot( timeplot, 
        m.bandflux(plotband, timeplot, zp=ptab['zp'][0], zpsys=ptab['zpsys'][0]),
             color='grey'
            )

plt.plot(ptab['time'], ptab['flux'], 'o')


Chidoff cut of 10 looks ok, but many objects do not have any good fits. Only for slsne? 
To avoid plotting too many lines, using the best fit for each SN seems reasonable.
We can kind of plot stuff, but would like some overview. 
Want to generate one overview plot for each SN. One panel for each band (2 or 3).
In the titles indicate name, whether it is in the base sample, some metrics about the fit. So loop through sne, plot these, also gather some statistics.
Among these, keep track of which templates / warp sn were used. Ideally we wish to only use a subset of them to limit how many fits are needed.
Maybe fine if not all can be well fit? Then we know ... some of the might be in the warp sample anyway, and will then be a template. 

Another sample plot needed is for every warp sn, to generate a bunch of simualted lightcurves, to check which look ok. 

In [ ]:
# Get models to study
def get_warpmodels( 
    warped_fits: dict, 
    max_chidof: int = 100,
    min_peakdof: int = 3,
    selectmode: None | Literal['bestsn','besttemplate'] = None,
    which_chidof : Literal['full','peak'] = 'full',
    ):
    """
    From a list of all sn/template warp combos fit to the sn,
    return a list of the best for study/plots.
    """

    # Collect some data for sorting and fitting
    fitsummary = [
        {
            'chisqdof': val['sncosmo_result']['chisq'] / val['sncosmo_result']['ndof'], 
            'base_sn': val['base_sn'],
            'base_template': val['base_template'],
            'warp_dps': val['dps_init'],
            'warp_ebv': val['hostebv'],
            'warp_success': val['success'],
            'warp_peakndof': val['ndof'],
            'warp_peakchidof': val['chidof'],
        } 
        for key, val in warped_fits.items()
    ]
    df = pd.DataFrame.from_dict(fitsummary)
    
    # Define rankval
    if which_chidof=='full':
        df['rankchi'] = df['chisqdof']
    elif which_chidof=='peak':
        df['rankchi'] = df['warp_peakchidof']
    df = df[df['rankchi']<max_chidof]
    df = df[df['warp_peakndof']>=min_peakdof]

    
    # Restrict to subset of best fit base sne or warp templates 
    if selectmode=='bestsn':
        df = df.sort_values('rankchi',ascending=True).groupby('base_sn').head(1)        
    elif selectmode=='besttemplate':
        df = df.sort_values('rankchi',ascending=True).groupby('base_template').head(1)  

    return {
        sn+'_'+t: warped_fits[sn+'_'+t] for sn, t in zip (df['base_sn'], df['base_template'])
    }


In [ ]:

# Set 1
#max_chidof, min_peakdof, selectmode, which_chidof = 30,5,'bestsn','full'
# No good - both sne matching fits as well as poor fits

# Set 2
# max_chidof, min_peakdof, selectmode, which_chidof = 30,5,'besttemplate','full'
# better, but not good

# Set 3
#max_chidof, min_peakdof, selectmode, which_chidof = 20,5,'besttemplate','full'
# matches are better, but many missing

# Set 4
#max_chidof, min_peakdof, selectmode, which_chidof = 10,5,'besttemplate','full'
# Fits generally better, but not many fits

# Set 5
# max_chidof, min_peakdof, selectmode, which_chidof = 10,5,'besttemplate','peak'
# yep, should definitely use peak

# Set 6
max_chidof, min_peakdof, selectmode, which_chidof = 1.,5,'bestsn','peak'
# Messed up by templates with repeated values. Cannot get slsn to work?

#max_chidof=30, 
#min_peakdof=5,
#selectmode = 'bestsn', # 'bestsn','besttemplate', None
#which_chidof = 'full' # full peak

# Semi
pdir = '/Users/jnordin/tmp/warptest/'

In [ ]:
# Make plots for each sn
# Record which base sn and templates were used
basesne, basetemps = [], []
for snname, fitdata in datacollect.items():
    print(snname)
    if not 'warped_fits' in fitdata:
        continue
    print(fitdata.keys())
    # Get photometry 
    tab = get_ztftable_from_ampel( snname, db, 
        redshift=float(df_bts[df_bts['ZTFID']==snname]['redshift'].iloc[0]),
        type = df_bts[df_bts['ZTFID']==snname]['type'].iloc[0] 
    )
    # Get warped fits
    warpcut = get_warpmodels( fitdata['warped_fits'],
                        max_chidof=max_chidof, 
                        min_peakdof=min_peakdof,
                        selectmode = selectmode,
                        which_chidof = which_chidof
                        )
    
    timeplot = np.arange( min(tab['time']), max(tab['time']), 1)
    bands = set(tab['band'])
    fig, axs = plt.subplots(1, len(bands))
    fig.suptitle( snname )

    for key, fitres in warpcut.items():    
        m = fitres['sncosmo_model']
        duplicate = False
        for k, band in enumerate(bands):
            ptab = tab[tab['band']==band]
            mflux = m.bandflux(band, timeplot, zp=ptab['zp'][0], zpsys=ptab['zpsys'][0])
            # Check if we have a uniqe prediction
            if len(set(mflux))<len(mflux):
                duplicate = True
#                break
            # Plots, assuming zp constant
            axs[k].plot( timeplot, mflux, color='grey' )

        if not duplicate:
            # Good model
            basesne.append( fitres['base_sn'] )
            basetemps.append( fitres['base_template'] )

    # Plot the SN light
    for k, band in enumerate(bands):
        ptab = tab[tab['band']==band]
        # SN photometry
        axs[k].plot(ptab['time'], ptab['flux'], 'o')

    fig.savefig( pdir + snname + '.png')
    plt.close(fig)
    #break
    

In [ ]:
Counter(basesne)

In [ ]:
Counter(basetemps)

In [ ]:
m.maxtime()

In [ ]:
2458607.6608912